# 03 - Model Training

This notebook demonstrates training different volatility forecasting models.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from src.models.har import HARModel
from src.models.garch import GARCHModel
from src.models.lightgbm_model import LightGBMModel
from src.evaluation.metrics import evaluate_forecast, qlike, mse

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Features

In [ ]:
# Load pre-computed features
FEATURES_DIR = Path("../data/features")

# For demo, create synthetic features
np.random.seed(42)
n = 2000

dates = pd.date_range('2015-01-01', periods=n, freq='B')
rv = np.abs(np.random.randn(n) * 0.001 + 0.0005)

df = pd.DataFrame({
    'date': dates,
    'rv': rv,
    'rv_d': np.roll(rv, 1),
    'rv_w': pd.Series(rv).rolling(5).mean().values,
    'rv_m': pd.Series(rv).rolling(22).mean().values,
    'jump': np.maximum(rv - rv * 0.85, 0),
})

# Create targets
df['y_h1'] = df['rv'].shift(-1)
df['y_h5'] = df['rv'].shift(-5)
df['y_h22'] = df['rv'].shift(-22)

# Log transform
df['log_rv'] = np.log(df['rv'] + 1e-10)
df['log_rv_d'] = np.log(df['rv_d'] + 1e-10)
df['log_rv_w'] = np.log(df['rv_w'] + 1e-10)
df['log_rv_m'] = np.log(df['rv_m'] + 1e-10)

df = df.iloc[22:-22].reset_index(drop=True)
print(f"Data shape: {df.shape}")
df.head()

## 2. Train/Val/Test Split

In [ ]:
# Split by date
train_end = '2017-12-31'
val_end = '2018-12-31'

train_df = df[df['date'] <= train_end]
val_df = df[(df['date'] > train_end) & (df['date'] <= val_end)]
test_df = df[df['date'] > val_end]

print(f"Train: {len(train_df)} ({train_df['date'].min()} to {train_df['date'].max()})")
print(f"Val: {len(val_df)} ({val_df['date'].min()} to {val_df['date'].max()})")
print(f"Test: {len(test_df)} ({test_df['date'].min()} to {test_df['date'].max()})")

## 3. HAR-RV Model

In [ ]:
# Features and target
feature_cols = ['rv_d', 'rv_w', 'rv_m', 'jump']
target_col = 'y_h1'

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_val = val_df[feature_cols]
y_val = val_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

In [ ]:
# Train HAR model
har_config = {
    'lags': [1, 5, 22],
    'include_jump': True,
    'use_log': True,
    'estimator': 'ols',
}

har_model = HARModel(horizon=1, config=har_config)
har_model.fit(X_train, y_train, X_val, y_val)

# Predict
y_pred_har = har_model.predict(X_test)

# Evaluate
metrics_har = evaluate_forecast(y_test.values, y_pred_har)
print("HAR-RV-J Results:")
print(f"  QLIKE: {metrics_har.qlike:.6f}")
print(f"  RMSE: {metrics_har.rmse:.6f}")
print(f"  R²: {metrics_har.r2:.4f}")

## 4. LightGBM Model

In [ ]:
# Train LightGBM
lgbm_config = {
    'n_estimators': 500,
    'max_depth': 5,
    'learning_rate': 0.05,
    'num_leaves': 15,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'early_stopping_rounds': 50,
    'verbose': -1,
}

lgbm_model = LightGBMModel(horizon=1, config=lgbm_config)
lgbm_model.fit(X_train, y_train, X_val, y_val)

# Predict
y_pred_lgbm = lgbm_model.predict(X_test)

# Evaluate
metrics_lgbm = evaluate_forecast(y_test.values, y_pred_lgbm)
print("LightGBM Results:")
print(f"  QLIKE: {metrics_lgbm.qlike:.6f}")
print(f"  RMSE: {metrics_lgbm.rmse:.6f}")
print(f"  R²: {metrics_lgbm.r2:.4f}")

## 5. Model Comparison

In [ ]:
# Compare predictions
comparison_df = pd.DataFrame({
    'Model': ['HAR-RV-J', 'LightGBM'],
    'QLIKE': [metrics_har.qlike, metrics_lgbm.qlike],
    'RMSE': [metrics_har.rmse, metrics_lgbm.rmse],
    'MAE': [metrics_har.mae, metrics_lgbm.mae],
    'R²': [metrics_har.r2, metrics_lgbm.r2],
})

comparison_df = comparison_df.sort_values('QLIKE')
comparison_df

In [ ]:
# Plot predictions
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(test_df['date'].values, y_test.values, label='Actual', color='black', linewidth=1.5)
ax.plot(test_df['date'].values, y_pred_har, label='HAR-RV-J', alpha=0.7)
ax.plot(test_df['date'].values, y_pred_lgbm, label='LightGBM', alpha=0.7)

ax.set_xlabel('Date')
ax.set_ylabel('Realized Volatility')
ax.set_title('Actual vs Predicted (H=1)')
ax.legend()

plt.tight_layout()
plt.show()

## 6. Residual Analysis

In [ ]:
# Residuals
resid_har = y_test.values - y_pred_har
resid_lgbm = y_test.values - y_pred_lgbm

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# HAR residuals histogram
axes[0, 0].hist(resid_har, bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(0, color='red', linestyle='--')
axes[0, 0].set_title('HAR Residuals Distribution')

# LightGBM residuals histogram
axes[0, 1].hist(resid_lgbm, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].axvline(0, color='red', linestyle='--')
axes[0, 1].set_title('LightGBM Residuals Distribution')

# HAR residuals over time
axes[1, 0].plot(test_df['date'].values, resid_har, linewidth=0.8)
axes[1, 0].axhline(0, color='red', linestyle='--')
axes[1, 0].set_title('HAR Residuals Over Time')

# LightGBM residuals over time
axes[1, 1].plot(test_df['date'].values, resid_lgbm, linewidth=0.8, color='orange')
axes[1, 1].axhline(0, color='red', linestyle='--')
axes[1, 1].set_title('LightGBM Residuals Over Time')

plt.tight_layout()
plt.show()

## 7. Feature Importance (LightGBM)

In [ ]:
# Get feature importance
importance = lgbm_model.get_feature_importance()

fig, ax = plt.subplots(figsize=(8, 5))
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=True)

ax.barh(importance_df['feature'], importance_df['importance'])
ax.set_xlabel('Importance')
ax.set_title('LightGBM Feature Importance')

plt.tight_layout()
plt.show()